# LLM Inference: GenAI Integrator Job Postings

This notebook classifies keyword-flagged job postings as GenAI *integrator* roles
using GPT-4o.  It produces two outputs per run:

- **`flagged_postings_gpt_predictions_<year>.csv`** — posting-level predictions
- **`firms_ever_genai_integrator_first_date_<year>.csv`** — firm-level first-integrator date

Run all cells top-to-bottom.  The only thing you need to change is the **Config** cell.

## 1. Imports

In [ ]:
import os
import re
import json
import time

import pandas as pd
import numpy as np
from openai import OpenAI


## 2. Config

Edit paths and output filenames here — nothing else needs changing.

In [ ]:
# ── Input parquet files (deduped, from deduplicate.ipynb) ─────────────────────
BASE_DIR   = "../data/Postings"
DIR_2025   = "../data/Postings/2025"

PARQUET_FILES_2022_2024 = [
    os.path.join(BASE_DIR, "SG-2022-WITHOUT-REPOSTS-W60D.parquet"),
    os.path.join(BASE_DIR, "SG-2023-WITHOUT-REPOSTS-W60D.parquet"),
    os.path.join(BASE_DIR, "SG-2024-WITHOUT-REPOSTS-W60D.parquet"),
]
PARQUET_FILE_2025 = os.path.join(DIR_2025, "2025_postings_merged_JAN_JUNE_WITHOUT_REPOSTS_W60D.parquet")

# ── Output paths ──────────────────────────────────────────────────────────────
# 2022-2024 run
OUT_PRED_2022_2024  = "../data/Firm Level/flagged_postings_gpt_predictions.csv"
OUT_FIRMS_2022_2024 = "../data/Firm Level/firms_ever_genai_integrator_first_date.csv"

# 2025 run
OUT_PRED_2025  = "../data/Firm Level/flagged_postings_gpt_predictions_2025.csv"
OUT_FIRMS_2025 = "../data/Firm Level/firms_ever_genai_integrator_first_date_2025.csv"

# Combined (2022-2025)
OUT_FIRMS_COMBINED = "../data/Firm Level/firms_ever_genai_integrator_first_date_2022_2025.csv"

# ── Industry filter ───────────────────────────────────────────────────────────
RECRUITMENT_INDUSTRIES = [
    "Recruitment and Staffing Services",
    "Employment and Staffing Services",
    "Employment and Recruitment Services",
    "Human Resources and Recruitment Services",
    "Online Employment Platforms",
    "Human Resources and Workforce Solutions",
    "Business Process Outsourcing Services",
]

# ── Model ─────────────────────────────────────────────────────────────────────
GPT_MODEL   = "gpt-4o"
SLEEP_S     = 0.1    # seconds between API calls (set 0 if rate limit is not a concern)
PRINT_EVERY = 50     # print posting details every N postings (set to 1 for full verbosity)


## 3. Load & Filter Data

In [ ]:
print("Loading 2022–2024 parquets …")
df_2022_2024 = pd.concat(
    [pd.read_parquet(f) for f in PARQUET_FILES_2022_2024],
    ignore_index=True,
)
df_2022_2024["post_date"] = pd.to_datetime(df_2022_2024["post_date"], errors="coerce")
print(f"  Combined shape: {df_2022_2024.shape}")

print("Loading 2025 parquet …")
df_2025 = pd.read_parquet(PARQUET_FILE_2025)
df_2025["post_date"] = pd.to_datetime(df_2025["post_date"], errors="coerce")
print(f"  2025 shape: {df_2025.shape}")


In [ ]:
def filter_recruitment(df: pd.DataFrame, industry_col: str = "rics_k400") -> pd.DataFrame:
    """Drop rows whose industry is a recruitment/staffing intermediary."""
    if industry_col not in df.columns:
        print(f"  [WARN] Column '{industry_col}' not found — skipping industry filter.")
        return df
    before = len(df)
    df = df[~df[industry_col].isin(RECRUITMENT_INDUSTRIES)].copy()
    print(f"  Dropped {before - len(df):,} recruitment-industry rows | Remaining: {len(df):,}")
    return df

print("Filtering 2022–2024 …")
df_2022_2024 = filter_recruitment(df_2022_2024)

print("Filtering 2025 …")
df_2025 = filter_recruitment(df_2025)


## 4. Keyword Pre-filter

Reduces the classification workload by keeping only postings that mention at least
one GenAI-related keyword in the title or description.  The LLM then decides whether
these are genuine *integrator* roles.

In [ ]:
GENAI_KEYWORDS = [
    "copilot", "claude", "gemini", "large language model", "llm",
    "generative ai", "chatgpt", "gen ai", "gpt", "langchain", "rag",
    "retrieval-augmented generation", "vector embedding", "vector database",
    "transformer", "prompt engineering", "prompt design", "llamaindex",
    "pinecone", "weaviate", "milvus", "openai api", "anthropic",
    "azure openai", "vertex ai", "huggingface", "retrievalqa",
]

# Word-boundary pattern to avoid substring contamination (e.g. "ragtime")
_KEYWORD_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in GENAI_KEYWORDS) + r")\b",
    re.IGNORECASE,
)


def build_flagged_set(
    df: pd.DataFrame,
    title_col: str = "jobtitle",
    desc_col: str = "description",
) -> pd.DataFrame:
    """
    Return rows containing at least one GenAI keyword in title or description.
    Adds 'keyword_flag' and 'matched_keywords' columns.
    """
    d = df.copy()
    d["_title_text"] = d[title_col].fillna("").astype(str)
    d["_desc_text"]  = d[desc_col].fillna("").astype(str)

    d["matched_keywords"] = d.apply(
        lambda row: list(set(
            _KEYWORD_PATTERN.findall(row["_title_text"])
            + _KEYWORD_PATTERN.findall(row["_desc_text"])
        )),
        axis=1,
    )
    d["keyword_flag"] = d["matched_keywords"].apply(lambda x: 1 if x else 0)
    d = d.drop(columns=["_title_text", "_desc_text"])

    flagged = d[d["keyword_flag"] == 1].copy()
    print(f"  Total flagged: {len(flagged):,} / {len(df):,} "
          f"({len(flagged)/len(df):.2%})")
    print("  Top matched keywords:")
    print(flagged["matched_keywords"].explode().value_counts().head(15).to_string())
    return flagged


In [ ]:
print("Flagging 2022–2024 …")
flagged_2022_2024 = build_flagged_set(df_2022_2024)

print("\nFlagging 2025 …")
flagged_2025 = build_flagged_set(df_2025)


## 5. GPT-4o Classifier

`classify_posting_with_gpt` sends a single posting to the API and returns a JSON
dict.  `gpt_label_flagged_postings` wraps this in a batch loop, saves results, and
derives the firm-level first-integrator date list.

In [ ]:
client = OpenAI()   # reads OPENAI_API_KEY from environment

SYSTEM_PROMPT = """
You are a classifier for job postings. Output ONLY compact JSON.

Role type definitions:
- integrator: builds/operates LLM systems (RAG, embeddings/vector DB, agents,
  LangChain/LlamaIndex, fine-tune/adapters, serving/inference, eval/guardrails,
  API integration).
- user: mainly uses LLM tools (ChatGPT, Gemini, Copilot, etc.) without building systems.
- both: does both integrator and user activities.
- none: neither.

Exclusions — NOT integrator if:
- Foundation-model research only
- Generic AI/ML roles without LLM system-building
- Developer roles at AI labs (e.g., OpenAI, DeepMind)
- Labeling/annotation roles

Department (choose one): Technology | Operations | Marketing | HR

Rules:
- If both integrator + user → role_type="both"
- Acronyms like "RAG" imply LLM context
- Prefer integrator=1 only when clear system-building signals appear
- JSON only; no prose

Output format:
{
  "integrator": 0|1,
  "user": 0|1,
  "role_type": "integrator"|"user"|"both"|"none",
  "department": "Technology"|"Operations"|"Marketing"|"HR",
  "confidence": 0.0-1.0,
  "reason": "short explanation"
}
"""

# Fallback returned when all retries are exhausted
_GPT_FAILURE_RESPONSE = {
    "integrator": 0, "user": 0, "role_type": "none",
    "department": "Operations", "confidence": 0.0, "reason": "API failure",
}


def classify_posting_with_gpt(
    title: str,
    description: str,
    model: str = GPT_MODEL,
    max_retries: int = 5,
) -> dict:
    """
    Classify a single posting via the OpenAI chat API.
    Returns a dict with keys: integrator, user, role_type, department, confidence, reason.
    """
    t = "" if title is None else str(title)
    d = "" if description is None else str(description)
    user_content = f"Job Title:\n{t}\n\nJob Description:\n{d}"

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_content},
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)

        except Exception as e:
            wait = min(2 ** attempt, 30)
            print(f"  [WARN] GPT call failed (attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(wait)

    return _GPT_FAILURE_RESPONSE


## 6. Sanity-check on a Small Sample

Run this before the full batch to verify the prompt is behaving as expected.
Tweak `n` as needed.

In [ ]:
def test_gpt_on_sample(
    flagged_df: pd.DataFrame,
    n: int = 5,
    title_col: str = "jobtitle",
    desc_col: str = "description",
    firm_col: str = "company",
    date_col: str = "post_date",
    id_col: str = "job_id",
    model: str = GPT_MODEL,
) -> None:
    """Print GPT classifications for the first *n* rows of flagged_df."""
    sample = flagged_df.head(n)
    print(f"Testing GPT on first {len(sample)} flagged postings …")
    print("=" * 80)

    for _, row in sample.iterrows():
        print(f"\nCompany   : {row.get(firm_col, 'N/A')}")
        print(f"Job ID    : {row.get(id_col, 'N/A')}")
        print(f"Job Title : {row.get(title_col, 'N/A')}")
        print(f"Post Date : {row.get(date_col, 'N/A')}")
        print("-" * 80)
        print(row.get(desc_col, ""))
        print("-" * 80)

        out = classify_posting_with_gpt(
            title=row.get(title_col, ""),
            description=row.get(desc_col, ""),
            model=model,
        )
        print(f"integrator : {out.get('integrator')}  |  role_type : {out.get('role_type')}  |  "
              f"user : {out.get('user')}  |  dept : {out.get('department')}  |  "
              f"conf : {out.get('confidence')}")
        print(f"reason     : {out.get('reason')}")
        print("=" * 80)


# Uncomment to run:
# test_gpt_on_sample(flagged_2022_2024, n=5)
# test_gpt_on_sample(flagged_2025, n=5)


## 7. Batch GPT Inference

In [ ]:
def gpt_label_flagged_postings(
    flagged_df: pd.DataFrame,
    out_pred_csv: str,
    out_firms_csv: str,
    firm_col: str = "company",
    date_col: str = "post_date",
    title_col: str = "jobtitle",
    desc_col: str = "description",
    id_col: str = "job_id",
    model: str = GPT_MODEL,
    sleep_s: float = SLEEP_S,
    print_every: int = PRINT_EVERY,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run GPT classification over all rows of flagged_df.

    Returns
    -------
    postings_df : posting-level DataFrame with GPT predictions appended
    firms_df    : firm-level DataFrame (company | first_integrator_post_date | n_integrator_postings)
    """
    required = [firm_col, date_col, title_col, desc_col]
    missing  = [c for c in required if c not in flagged_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = flagged_df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    keep = [c for c in [id_col, firm_col, date_col, title_col, desc_col,
                         "keyword_flag", "matched_keywords"] if c in df.columns]
    df = df[keep].copy()

    n_total = len(df)
    print(f"Flagged postings to classify: {n_total:,}")
    print("Running GPT classification …")
    print("=" * 80)

    records = []
    for k, (_, row) in enumerate(df.iterrows(), start=1):
        title   = row.get(title_col, "")
        desc    = row.get(desc_col, "")
        company = row.get(firm_col, "N/A")
        job_id  = row.get(id_col, "N/A")
        post_date = row.get(date_col, "N/A")

        if print_every and (k % print_every == 0):
            print(f"\nCompany   : {company}")
            print(f"Job ID    : {job_id}  |  Post Date : {post_date}")
            print(f"Job Title : {title}")
            print("-" * 80)

        out = classify_posting_with_gpt(title=title, description=desc, model=model)

        integrator = int(out.get("integrator", 0))
        records.append({
            "is_integrator_gpt": integrator,
            "gpt_role_type":     out.get("role_type", "none"),
            "gpt_user":          int(out.get("user", 0)),
            "gpt_department":    out.get("department", "Operations"),
            "gpt_confidence":    float(out.get("confidence", 0.0) or 0.0),
            "gpt_reason":        str(out.get("reason", "")).strip(),
        })

        if print_every and (k % print_every == 0):
            print(f"integrator : {integrator}  |  role_type : {records[-1]['gpt_role_type']}  |  "
                  f"conf : {records[-1]['gpt_confidence']:.2f}")
            print(f"reason     : {records[-1]['gpt_reason']}")
            print("=" * 80)

        if k % 100 == 0:
            print(f"  … classified {k:,} / {n_total:,}")

        if sleep_s > 0:
            time.sleep(sleep_s)

    pred_df = pd.concat([df.reset_index(drop=True),
                          pd.DataFrame(records)], axis=1)

    os.makedirs(os.path.dirname(out_pred_csv) or ".", exist_ok=True)
    pred_df.to_csv(out_pred_csv, index=False)
    print(f"\nSaved posting-level predictions → {out_pred_csv}")

    # ── Firm-level: ever-integrator + first date ───────────────────────────────
    integrator_rows = pred_df[pred_df["is_integrator_gpt"] == 1].dropna(subset=[firm_col, date_col])

    if integrator_rows.empty:
        firms_df = pd.DataFrame(columns=[firm_col, "first_integrator_post_date", "n_integrator_postings"])
        print("  No integrator postings found — saving empty firm list.")
    else:
        firms_df = (
            integrator_rows
            .groupby(firm_col, as_index=False)
            .agg(
                first_integrator_post_date=(date_col, "min"),
                n_integrator_postings=("is_integrator_gpt", "sum"),
            )
            .sort_values("first_integrator_post_date")
        )

    os.makedirs(os.path.dirname(out_firms_csv) or ".", exist_ok=True)
    firms_df.to_csv(out_firms_csv, index=False)
    print(f"Saved firm-level integrator list    → {out_firms_csv}")

    # ── Summary ────────────────────────────────────────────────────────────────
    n_firms_flagged  = pred_df[firm_col].nunique(dropna=True)
    n_firms_treated  = firms_df[firm_col].nunique(dropna=True)
    pct = n_firms_treated / n_firms_flagged if n_firms_flagged else float("nan")

    print("\n─── Summary ─────────────────────────────────────────────────────")
    print(f"Unique firms in flagged set : {n_firms_flagged:,}")
    print(f"Unique firms ever-integrator: {n_firms_treated:,}  ({pct:.2%} of flagged firms)")
    print(f"Integrator postings (GPT=1) : "
          f"{int(pred_df['is_integrator_gpt'].sum()):,} / {n_total:,} "
          f"({pred_df['is_integrator_gpt'].mean():.2%})")

    return pred_df, firms_df


## 8. Run: 2022–2024

In [ ]:
postings_pred_2022_2024, firms_2022_2024 = gpt_label_flagged_postings(
    flagged_df=flagged_2022_2024,
    out_pred_csv=OUT_PRED_2022_2024,
    out_firms_csv=OUT_FIRMS_2022_2024,
)


## 9. Run: 2025

In [ ]:
postings_pred_2025, firms_2025 = gpt_label_flagged_postings(
    flagged_df=flagged_2025,
    out_pred_csv=OUT_PRED_2025,
    out_firms_csv=OUT_FIRMS_2025,
)


## 10. Consolidate Firm Lists (2022–2025)

Stack both firm-level CSVs, keep the earliest integrator date per firm, and sum
posting counts.

In [ ]:
def consolidate_firm_lists(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    firm_col: str = "company",
    date_col: str = "first_integrator_post_date",
    count_col: str = "n_integrator_postings",
    out_path: str = OUT_FIRMS_COMBINED,
) -> pd.DataFrame:
    """Stack two firm-level CSVs, keep earliest date and sum posting counts per firm."""
    combined = pd.concat([df1, df2], ignore_index=True)
    combined[date_col] = pd.to_datetime(combined[date_col], errors="coerce")

    print(f"Total rows before consolidation : {len(combined):,}")
    print(f"Unique firms before             : {combined[firm_col].nunique():,}")

    final = (
        combined
        .dropna(subset=[firm_col, date_col])
        .groupby(firm_col, as_index=False)
        .agg(
            **{date_col:  (date_col,  "min"),
               count_col: (count_col, "sum")},
        )
        .sort_values(date_col)
    )

    print(f"Unique firms after consolidation: {final[firm_col].nunique():,}")

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    final.to_csv(out_path, index=False)
    print(f"Saved consolidated firm list → {out_path}")
    return final


final_firms = consolidate_firm_lists(firms_2022_2024, firms_2025)
final_firms.head(10)
